# Colab: doğrulanmış dense artefakt üretimi

Bu notebook Qwen dense sayfa vektörlerini sabit kod ve kaynak indeks commit'leriyle üretir. Tamamlanan her modeli ayrı Hugging Face Dataset'e yükler; üretim indeksine veya cevap eşiğine dokunmaz.

Başarı ölçütü: her model için `embeddings.npy`, `dense.json` ve çıktıdaki değişmez Hub commit SHA'sı.

In [ ]:
from pathlib import Path

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
SOURCE_REPO = 'barandincoguz/belge-gozu-index'
SOURCE_REVISION = '700ac324fffefb22de02c8e90347b31185547948'
CODE_REPOSITORY = 'https://github.com/barandincoguz/belge-gozu.git'
CODE_REVISION = '80783c9534bf09caaad9752675e2497009c2c028'
ARTIFACT_REPO = 'barandincoguz/belge-gozu-semantic-artifacts'
MODELS = ['qwen3-embedding-4b', 'qwen3-embedding-8b']

if not HF_TOKEN:
    raise RuntimeError('Colab Secrets içine write yetkili HF_TOKEN ekleyin.')
for name, value in {'SOURCE_REVISION': SOURCE_REVISION, 'CODE_REVISION': CODE_REVISION}.items():
    if len(value) != 40 or value.startswith('PASTE_'):
        raise ValueError(f'{name} gerçek bir 40 karakterli commit SHA olmalı.')

## Kalıcı checkpoint ve GPU ön kontrolü

Drive checkpoint'i Colab oturumu kesilirse aynı modelin kaldığı batch'ten sürmesini sağlar. 8B modeli 24 GB altındaki GPU'da çalıştırılmaz; model sessizce nicemlenmez veya küçültülmez.

In [ ]:
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Runtime > Change runtime type ile GPU seçin.')
gpu = torch.cuda.get_device_properties(0)
GPU_BYTES = gpu.total_memory
print({'gpu': gpu.name, 'vram_gib': round(GPU_BYTES / 1024**3, 1)})

drive.mount('/content/drive')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/belge-gozu-dense-artifacts')
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

## Sabit kod ve kaynak indeks

Kod yalnız verilen Git commit'inden çalışır. Kaynak Dataset'ten sadece dense kodlamanın kullandığı `page_texts.parquet` indirilir.

In [ ]:
import subprocess
import sys


def run(command, *, cwd=None):
    print('$', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

WORKTREE = Path('/content/belge-gozu')
if WORKTREE.exists():
    run(['git', 'fetch', 'origin'], cwd=WORKTREE)
else:
    run(['git', 'clone', CODE_REPOSITORY, str(WORKTREE)])
run(['git', 'checkout', '--detach', CODE_REVISION], cwd=WORKTREE)
run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-e', '.', 'transformers>=5.3,<6'],
    cwd=WORKTREE,
)

def download_source_index():
    from huggingface_hub import snapshot_download

    source_root = Path('/content/source-index')
    snapshot_download(
        repo_id=SOURCE_REPO, repo_type='dataset', revision=SOURCE_REVISION, token=HF_TOKEN,
        allow_patterns=['index/page_texts.parquet'], local_dir=source_root,
    )
    return source_root / 'index'

INDEX_DIR = download_source_index()
if not (INDEX_DIR / 'page_texts.parquet').is_file():
    raise RuntimeError('Sabit kaynak commit page_texts.parquet sağlamadı.')

## Model başına üret, doğrula ve yükle

Bir model tamamlanmadan Hub'a yüklenmez. `in_progress` çıktısı alınırsa bu hücre aynı Drive checkpoint'iyle yeniden çalıştırılır. Her başarılı yüklemenin yazdığı commit SHA'yı yerel değerlendirme için kaydedin.

In [ ]:
import os

os.chdir(WORKTREE)
sys.path.insert(0, str(WORKTREE / 'src'))

def build_and_publish():
    from belge_gozu.bench.dense_artifact_hub import push_dense_artifact
    from belge_gozu.bench.dense_artifacts import DenseArtifactExpectation, sha256_file
    from belge_gozu.retrieval.dense import DENSE_MODELS
    from belge_gozu.retrieval.hybrid import load_page_texts

    page_texts = load_page_texts(INDEX_DIR)
    page_texts_hash = sha256_file(INDEX_DIR / 'page_texts.parquet')
    published = {}
    for model_key in MODELS:
        if model_key == 'qwen3-embedding-8b' and GPU_BYTES < 24 * 1024**3:
            raise RuntimeError('qwen3-embedding-8b için en az 24 GiB GPU belleği gerekir.')
        run([
            sys.executable, 'scripts/build_dense_artifacts.py', '--index-dir', str(INDEX_DIR),
            '--source-repo', SOURCE_REPO, '--source-revision', SOURCE_REVISION,
            '--model', model_key, '--artifact-root', str(ARTIFACT_ROOT), '--device', 'cuda',
            '--batch-size', '1',
        ], cwd=WORKTREE)
        artifact_dir = ARTIFACT_ROOT / model_key
        if not (artifact_dir / 'dense.json').is_file():
            print({'model': model_key, 'status': 'in_progress'})
            break
        expectation = DenseArtifactExpectation(
            DENSE_MODELS[model_key], list(page_texts), page_texts_hash
        )
        published[model_key] = push_dense_artifact(
            artifact_dir, ARTIFACT_REPO, model_key, expectation, token=HF_TOKEN
        )
        print({'model': model_key, 'artifact_hub_commit': published[model_key]})
    return published

published = build_and_publish()
published

## Sonraki adım

Yerelde her model için yazılan Hub commit SHA ile `scripts/pull_dense_artifacts.py` çalıştırın. Ardından `scripts/eval_semantic_coverage.py` ile offline ölçümü üretin. Bu notebook'un başarılı olması tek başına üretim açılışı veya eşik kalibrasyonu değildir.